# 🦜 LangChain Agents

**Topics Covered:**
1. Basic Agent Creation
2. Messages
3. Models (Groq AI)
4. Tools
5. Student Helper Bot

> **Model used:** `groq:llama-3.3-70b-versatile` via LangChain's Groq integration

## ⚙️ Installation
Run this once to install the required packages.

In [ ]:
!pip install langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.4 MB/s eta 0:00:00


## 🔑 Set Your Groq API Key
Get a free API key from [https://console.groq.com](https://console.groq.com)

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "MY-api-key"  # ← Replace with your key

---
## 📌 Topic 1: Basic Agent Creation

An **agent** = **Model** (brain) + **Tools** (hands)

- `create_agent()` → builds the agent
- `invoke()` → runs it with a message

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# --- Define a simple tool ---
@tool
def say_hello(name: str) -> str:
    """Say hello to someone."""
    return f"Hello, {name}! Welcome to LangChain."

# --- Create the agent using Groq model ---
agent = create_agent(
    "groq:llama-3.3-70b-versatile",
    tools=[say_hello]
)

# --- Run the agent ---
result = agent.invoke({
    "messages": [{"role": "user", "content": "Say hello to Rajan"}]
})

print(result["messages"][-1].content)
print(agent)

I'm happy to help you with any questions or topics you'd like to discuss. Is there something specific you'd like to chat about, or do you have any questions for me?


---
## 📌 Topic 2: Messages

Agents communicate through **messages**. There are 3 types:

| Role | Purpose |
|---|---|
| `"user"` | The human's message |
| `"assistant"` | The model's reply |
| `"system"` | Instructions that shape the model's behavior |

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_fact(topic: str) -> str:
    """Return a fun fact about a topic."""
    return f"Fun fact about {topic}: It is absolutely fascinating!"

# system_prompt sets a system message for the agent
agent = create_agent(
    "groq:qwen/qwen3-32b", # Changed model to llama-3.1-70b-versatile
    tools=[get_fact],
    system_prompt=(
        "You are a friendly teacher. "
        "Your primary goal is to help students learn by providing facts using the 'get_fact' tool. "
        "When a user asks for a fact, always use the 'get_fact' tool. "
        "If asked for multiple facts, use the 'get_fact' tool for each request in the conversation. "
        "Keep your answers short and simple."
    )
)

# --- Single user message ---
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Use the get_fact tool to tell me a fact about space"}
    ]
})
print("Single message reply:")
print(result["messages"][-1].content)

print()

# --- Multi-turn conversation (chat history) ---
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Use the get_fact tool to tell me a fact about space"},
        {"role": "user",  "content": "Now use the get_fact tool to tell me a fact about the moon."}  # Changed topic for follow-up
    ]
})
print("Multi-turn reply:")
print(result["messages"][-1].content)

Single message reply:
Here's your space fact: **It is absolutely fascinating!** 🌌

Let me know if you'd like another fun fact!

Multi-turn reply:
🌕 Fun fact about the moon: It is absolutely fascinating! Let me know if you'd like more details.


---
## 📌 Topic 3: Models (Groq AI)

The **model** is the brain of the agent. We use **Groq** for fast inference.

| Approach | When to Use |
|---|---|
| `"groq:model-name"` string | Simple and quick |
| `ChatGroq(...)` object | When you need to control temperature, tokens, etc. |

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq

@tool
def add_numbers(a: int, b: int) -> str:
    """Add two numbers together."""
    return f"{a} + {b} = {a + b}"

# --- Option A: Simple string (easy for beginners) ---
agent_simple = create_agent(
    "groq:llama-3.3-70b-versatile",
    tools=[add_numbers]
)

# --- Option B: ChatGroq object (full control) ---
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,      # 0 = focused/deterministic, 1 = creative
    max_tokens=300
)
agent_detailed = create_agent(model, tools=[add_numbers])

# Test both
result_a = agent_simple.invoke({
    "messages": [{"role": "user", "content": "What is 25 + 37?"}]
})
print("Option A (string):", result_a["messages"][-1].content)

result_b = agent_detailed.invoke({
    "messages": [{"role": "user", "content": "What is 100 + 250?"}]
})
print("Option B (ChatGroq):", result_b["messages"][-1].content)

Option A (string): The answer is 62.
Option B (ChatGroq): The answer is 350.


---
## 📌 Topic 4: Tools

**Tools** let the agent DO things, not just talk.

- Use `@tool` decorator to define a tool
- The **docstring** is what the model reads to understand the tool
- You can give the agent **multiple tools** and it picks the right one

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {
        "Chennai": "35°C, Sunny",
        "Mumbai":  "30°C, Cloudy",
        "Delhi":   "28°C, Windy",
    }
    return weather_data.get(city, "Weather data not available for this city.")

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    rates = {"USD_to_INR": 83.0, "INR_to_USD": 0.012}
    key = f"{from_currency}_to_{to_currency}"
    if key in rates:
        converted = amount * rates[key]
        return f"{amount} {from_currency} = {converted:.2f} {to_currency}"
    return "Conversion not supported."

# Agent with multiple tools — it picks the right one automatically
agent = create_agent(
    "groq:llama-3.3-70b-versatile",
    tools=[get_weather, convert_currency]
)

# Test weather tool
result = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in Chennai?"}]
})
print("Weather:", result["messages"][-1].content)

# Test currency tool
result = agent.invoke({
    "messages": [{"role": "user", "content": "Convert 100 USD to INR"}]
})
print("Currency:", result["messages"][-1].content)

Weather: The current weather in Chennai is 35°C and sunny.
Currency: The conversion result is 8300 INR.


---
## 📌 Topic 5: Student Helper Bot

Putting it all together:

✅ **Groq model** (ChatGroq)  
✅ **System prompt** (sets behavior)  
✅ **Multiple tools** (search, math, tips)  
✅ **User messages** (questions from students)  

**Scenario:** A bot that helps students study 📚

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq

# ── Step 1: Define the Model ──────────────────────────────────
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=300
)

# ── Step 2: Define Tools ──────────────────────────────────────
@tool
def search_topic(topic: str) -> str:
    """Search and return study material on a topic."""
    study_material = {
        "python":    "Python is a high-level programming language with simple syntax.",
        "langchain": "LangChain is a framework for building LLM-powered applications.",
        "ai":        "Artificial Intelligence is the simulation of human intelligence by machines.",
    }
    return study_material.get(topic.lower(), f"No material found for '{topic}'.")

@tool
def solve_math(expression: str) -> str:
    """Solve a basic math expression like '10 + 5' or '100 / 4'."""
    try:
        answer = eval(expression)
        return f"Answer: {answer}"
    except Exception:
        return "Could not solve. Try simple expressions like '10 + 5'."

@tool
def study_tip(subject: str) -> str:
    """Give a study tip for a subject."""
    tips = {
        "math":        "Practice problems daily. Start with easy ones and build up.",
        "programming": "Write code every day. Learn by building small projects.",
        "english":     "Read books and write summaries to improve your writing.",
    }
    return tips.get(subject.lower(), "Break topics into small chunks and review regularly.")

# ── Step 3: Create the Agent ──────────────────────────────────
student_bot = create_agent(
    model=model,
    tools=[search_topic, solve_math, study_tip],
    system_prompt=(
        "You are a helpful student assistant. "
        "Help students learn by searching topics, solving math, and giving study tips. "
        "Keep your answers short and friendly."
    )
)

# ── Step 4: Ask Questions ─────────────────────────────────────
questions = [
    "What is Python?",
    "Solve this: 25 * 4",
    "Give me a study tip for math",
]

for question in questions:
    result = student_bot.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    print(f"Q: {question}")
    print(f"A: {result['messages'][-1].content}")
    print()

Q: What is Python?
A: Python is a high-level, interpreted programming language that is widely used for web development, scientific computing, data analysis, artificial intelligence, and more. Its simplicity and readability make it an ideal language for beginners and experts alike. If you're interested in learning more about Python, I can provide you with some study materials and tips. Would you like that?

Q: Solve this: 25 * 4
A: The answer is 100.

Q: Give me a study tip for math
A: I see you're looking for a study tip for math. Here's one: practice problems daily, starting with easy ones and building up to more challenging ones. This will help you build confidence and fluency in math.



In [ ]:
import os
from langchain.agents import create_agent
from langchain.tools import tool

os.environ["GROQ_API_KEY"] = "****insert-api-key"  # ← Replace with your key


# ── Define Tools ──────────────────────────────────────────────

@tool
def get_student_marks(student_name: str) -> str:
    """Get the exam marks for a student by name."""
    marks_db = {
        "Rajan":  {"Math": 88, "Science": 75, "English": 90},
        "Priya":  {"Math": 95, "Science": 89, "English": 78},
        "Kumar":  {"Math": 60, "Science": 55, "English": 70},
    }
    student = marks_db.get(student_name)
    if student:
        return str(student)
    return f"No record found for student '{student_name}'."


@tool
def calculate_average(marks_str: str) -> str:
    """Calculate the average from a dictionary of subject marks.
    Input should be a dict string like: {'Math': 88, 'Science': 75, 'English': 90}
    """
    import ast
    try:
        marks = ast.literal_eval(marks_str)
        values = list(marks.values())
        avg = sum(values) / len(values)
        return f"Average marks: {avg:.2f}"
    except Exception as e:
        return f"Error calculating average: {e}"


@tool
def check_pass_or_fail(average_marks: str) -> str:
    """Check if a student passed or failed based on their average marks.
    Passing mark is 60. Input should be a string like: 'Average marks: 84.33'
    """
    try:
        avg = float(average_marks.replace("Average marks:", "").strip())
        if avg >= 60:
            return f"✅ PASSED with average {avg:.2f}"
        else:
            return f"❌ FAILED with average {avg:.2f}. Needs improvement."
    except Exception as e:
        return f"Error checking result: {e}"


# ── Create the Agent ──────────────────────────────────────────

agent = create_agent(
    "groq:qwen/qwen3-32b",
    tools=[get_student_marks, calculate_average, check_pass_or_fail],
    system_prompt=(
        "You are a school result analyser. "
        "When asked about a student's result, follow these steps:\n"
        "1. Get the student's marks\n"
        "2. Calculate their average\n"
        "3. Check if they passed or failed\n"
        "Then give a short final summary."
    )
)


# ── Stream to show the ReAct Loop Step by Step ───────────────

print("=" * 55)
print("  ReAct Loop — LangChain + Groq AI")
print("=" * 55)
print("Question: Check the result for student Rajan")
print("-" * 55)

step = 1
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Check the result for student Rajan"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]

    # 🔧 ACT — agent is calling a tool
    if hasattr(latest, "tool_calls") and latest.tool_calls:
        for tc in latest.tool_calls:
            print(f"\n🧠 THINK + ACT (Step {step})")
            print(f"   Tool called : {tc['name']}")
            print(f"   Input       : {tc['args']}")
            step += 1

    # 👁️ OBSERVE — tool result came back
    elif hasattr(latest, "type") and latest.type == "tool":
        print(f"   👁️ Observation : {latest.content}")

    # ✅ FINISH — final answer
    elif latest.content and latest.type == "ai":
        print(f"\n✅ FINAL ANSWER:")
        print(f"   {latest.content}")

print("-" * 55)
print("ReAct loop complete!")

  ReAct Loop — LangChain + Groq AI
Question: Check the result for student Rajan
-------------------------------------------------------

🧠 THINK + ACT (Step 1)
   Tool called : get_student_marks
   Input       : {'student_name': 'Rajan'}
   👁️ Observation : {'Math': 88, 'Science': 75, 'English': 90}

🧠 THINK + ACT (Step 2)
   Tool called : calculate_average
   Input       : {'marks_str': "{'Math': 88, 'Science': 75, 'English': 90}"}
   👁️ Observation : Average marks: 84.33

🧠 THINK + ACT (Step 3)
   Tool called : check_pass_or_fail
   Input       : {'average_marks': 'Average marks: 84.33'}
   👁️ Observation : ✅ PASSED with average 84.33

✅ FINAL ANSWER:
   **Rajan's Exam Results:**  
- **Math:** 88  
- **Science:** 75  
- **English:** 90  
**Average:** 84.33  
**Status:** ✅ **PASSED**  

Let me know if you need further details!
-------------------------------------------------------
ReAct loop complete!


---
## ✅ Summary

| Topic | Key Concept |
|---|---|
| Basic Agent | `create_agent()` + `invoke()` |
| Messages | `user`, `assistant`, `system` roles |
| Models | String shortcut vs `ChatGroq()` object |
| Tools | `@tool` decorator — docstring = description |
| Integrated | All combined into a Student Helper Bot |

> 🚀 **Next Steps:** Explore streaming, structured output, and memory!